# v4.1 Adaptive Reasoning System — Phase 6: LLM Integration

This notebook adds full LLM capabilities:
- **Semantic parsing** from natural language (no pre-parsed JSON)
- **Code generation** solver for coding tasks
- **Critique & repair** via LLM
- **Solver formulation** proposals

**Requires:** A100/H100 GPU with ≥40GB VRAM for 7-8B model inference.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
PROJECT_ROOT = '/content/drive/MyDrive/agi'
sys.path.insert(0, PROJECT_ROOT)

# Pin ortools<9.12 to avoid protobuf>=6 conflict with Colab's tensorflow/google-ai
!pip install -q torch transformers accelerate bitsandbytes "ortools>=9.7,<9.12" faiss-cpu pydantic rich numpy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 35.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 5.26.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 5.26.1 which is incompatible.


In [2]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 1. Initialize LLM Parser

Choose your model (uncomment one):
- `mistralai/Mistral-7B-Instruct-v0.3`
- `meta-llama/Llama-3.1-8B-Instruct`
- `Qwen/Qwen2.5-7B-Instruct`

In [3]:
#MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
# MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

from v4_1_reasoning_system.parser.llm_parser import LLMParser

parser = LLMParser(
    model_name=MODEL_NAME,
    device=device,
    load_in_4bit=True,
    max_new_tokens=1024,
)
print(f'Parser initialized with {MODEL_NAME} (lazy-loaded)')

Parser initialized with meta-llama/Llama-3.1-8B-Instruct (lazy-loaded)


In [4]:
# Test parsing a natural-language problem
task = parser.parse(
    "I need to schedule 5 workers across 3 shifts. Each shift needs at least "
    "1 worker, no worker can do more than 2 shifts, and I want to minimize "
    "total overtime cost. Workers A and B cannot be on the same shift."
)

print(f'Domain: {task.domain}')
print(f'Description: {task.description}')
print(f'Entities: {task.entities}')
print(f'Constraints: {len(task.constraints)}')
for c in task.constraints:
    print(f'  - [{c.ctype.value}] {c.name}: {c.description}')
if task.objective:
    print(f'Objective: {task.objective.sense.value} {task.objective.description}')
print(f'Ambiguities: {len(task.ambiguities)}')

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Domain: DomainGuess.SCHEDULING
Description: Schedule 5 workers across 3 shifts with constraints
Entities: ['workers', 'shifts']
Constraints: 3
  - [hard] min workers per shift: each shift needs at least 1 worker
  - [hard] max shifts per worker: no worker can do more than 2 shifts
  - [hard] workers A and B cannot be on the same shift: workers A and B cannot be on the same shift
Objective: minimize minimize total overtime cost
Ambiguities: 1


## 2. Full Controller with LLM

Initialize the controller with the LLM parser and codegen solver.

In [5]:
from v4_1_reasoning_system.orchestration.controller import ReasoningController
from v4_1_reasoning_system.solvers.llm_codegen_solver import LLMCodegenSolver

LATENT_DIM = 128
ACTION_DIM = 32

# LLM codegen solver (shares the same model as parser)
codegen_solver = LLMCodegenSolver(
    model_name=MODEL_NAME,
    device=device,
    load_in_4bit=True,
)

controller = ReasoningController(
    parser=parser,
    use_learned_router=False,
    max_iterations=5,
    max_time_seconds=180.0,
    device=device,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
)

# Register the LLM-backed codegen solver
controller.register_solver('llm_codegen', codegen_solver)
print('Controller ready with full LLM integration')

Controller ready with full LLM integration


## 3. Solve from Natural Language (No Pre-Parsing)

In [6]:
# Scheduling problem — fully natural language
result = controller.solve(
    "Schedule 4 maintenance jobs on a single machine. "
    "Job A takes 3 hours, Job B takes 5 hours and must come after A, "
    "Job C takes 2 hours and can run anytime, "
    "Job D takes 4 hours and must come after both B and C. "
    "Minimize the total completion time."
)

print(f'Valid: {result["valid"]}, Score: {result["score"]:.3f}')
print(f'Domain: {result["domain"]}')
print(f'Iterations: {result["iterations"]}')
print(f'Solution: {result["solution"]}')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Valid: True, Score: 1.000
Domain: scheduling
Iterations: 1
Solution: {'job_A': {'start': 0, 'end': 3}, 'job_B': {'start': 3, 'end': 8}, 'job_C': {'start': 0, 'end': 2}, 'job_D': {'start': 8, 'end': 12}}


In [7]:
# Print reasoning trajectory
for line in result['logs']:
    print(line)

Parsed task: Domain: scheduling | Entities: 4 | Hard constraints: 2 | Soft constraints: 0 | Objective: minimize | Ambiguities: 0
Structured data: type=scheduling, keys=['type', 'jobs', 'horizon', 'resources']
Initial state encoded (latent dim=128)

--- Iteration 0 ---
Generated 4 candidates
Rule router selected: idx=0
  Mode: global_opt, Budget: high, Hint: feasibility_priority
  Solver: success=True, score=12.000, feasible=True, elapsed=0.05s
  Verifier: valid=True, score=1.000, passed=5/5
Verified success at iteration 0!

=== Done: 1 steps, score=1.000, elapsed=16.82s ===


## 4. Coding Task with LLM Code Generation

In [8]:
coding_result = controller.solve(
    "Write a Python function that finds the longest common subsequence "
    "of two strings. Print the result as JSON with keys 'length' and 'subsequence' "
    "for inputs 'ABCBDAB' and 'BDCAB'."
)

print(f'Valid: {coding_result["valid"]}')
print(f'Iterations: {coding_result["iterations"]}')
if isinstance(coding_result['solution'], dict) and 'code' in coding_result['solution']:
    print('\nGenerated code:')
    print(coding_result['solution']['code'])
    if 'output' in coding_result['solution']:
        print(f'\nOutput: {coding_result["solution"]["output"]}')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Valid: False
Iterations: 5


## 5. Optimization Problem from Natural Language

In [9]:
opt_result = controller.solve(
    "I have a delivery truck with capacity 50kg. I need to choose which packages to load. "
    "Package 1: 10kg, value $15. Package 2: 20kg, value $25. Package 3: 15kg, value $20. "
    "Package 4: 25kg, value $35. Package 5: 8kg, value $12. "
    "Maximize total delivery value without exceeding capacity."
)

print(f'Valid: {opt_result["valid"]}, Score: {opt_result["score"]:.3f}')
print(f'Solution: {opt_result["solution"]}')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Valid: True, Score: 1.000
Solution: {'selected_items': [0, 2, 3]}


## 6. Load Trained Checkpoint + LLM

Combine a pre-trained world model & router with live LLM parsing.

In [10]:
# Load previous checkpoint (from notebook 02)
CHECKPOINT_DIR = '/content/drive/MyDrive/agi/checkpoints/v4_1_trained'

from pathlib import Path
if Path(CHECKPOINT_DIR).exists():
    controller.load_checkpoint(CHECKPOINT_DIR)
    controller.use_learned_router = True
    print(f'Loaded checkpoint from {CHECKPOINT_DIR}')
    print(f'Replay buffer: {len(controller.replay_buffer)} trajectories')
    print('Router: EBM (learned)')
else:
    print(f'No checkpoint at {CHECKPOINT_DIR}, using rule-based router')

Loaded checkpoint from /content/drive/MyDrive/agi/checkpoints/v4_1_trained
Replay buffer: 93 trajectories
Router: EBM (learned)


In [11]:
# Solve with trained router + LLM parsing
trained_result = controller.solve(
    "Assign 6 students to 3 project groups. Each group must have exactly 2 students. "
    "Students Alice and Bob cannot be in the same group. "
    "Minimize the maximum skill gap within each group."
)

print(f'Valid: {trained_result["valid"]}')
print(f'Solution: {trained_result["solution"]}')
for line in trained_result['logs']:
    print(line)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Valid: True
Solution: {'type': 'hierarchical_plan', 'steps': [{'subgoal': 'Assign students to groups', 'solved': True, 'solution': {'subgoal': 'Assign students to groups', 'status': 'completed', 'dependencies_met': True}}], 'complete': True}
Parsed task: Domain: planning | Entities: 4 | Hard constraints: 2 | Soft constraints: 0 | Objective: minimize | Ambiguities: 1
Structured data: type=planning, keys=['type', 'subgoals']
Initial state encoded (latent dim=128)

--- Iteration 0 ---
Generated 4 candidates
EBM router selected: idx=0 (E=-0.315)
  All energies: ['-0.315', '0.181', '0.557', '-0.182']
  Mode: hierarchical, Budget: medium, Hint: depth_2
  Solver: success=True, score=1.000, feasible=True, elapsed=0.00s
  Verifier: valid=True, score=1.000, passed=3/4
Verified success at iteration 0!

=== Done: 1 steps, score=1.000, elapsed=14.20s ===


## 7. Save Updated Checkpoint

In [12]:
SAVE_DIR = '/content/drive/MyDrive/agi/checkpoints/v4_1_llm_full'
controller.save_checkpoint(SAVE_DIR)
print(f'Checkpoint saved to {SAVE_DIR}')

Checkpoint saved to /content/drive/MyDrive/agi/checkpoints/v4_1_llm_full
